# فاز دوم: احتمال و توزیع‌های احتمالی

## پروژه NHANES 2021–2023

در این فاز، با استفاده از نمونه تحلیلی نهایی حاصل از فاز اول، مفاهیم احتمال و توزیع‌های احتمالی را در داده‌های واقعی بررسی می‌کنیم. هدف، تعریف پیشامدهای مرتبط با سؤال پژوهش، محاسبه احتمال‌های تجربی و شرطی، بررسی استقلال، مطالعه توزیع متغیرها و محاسبه امید ریاضی، واریانس و کوواریانس است.

## ۱. آماده‌سازی محیط و بارگذاری داده‌ها

در این بخش، کتابخانه‌های موردنیاز را وارد کرده و نمونه تحلیلی نهایی ذخیره‌شده در فاز اول را بارگذاری می‌کنیم. این داده شامل شرکت‌کنندگانی است که اطلاعات معتبر BMI و میانگین فشار خون سیستولیک آن‌ها در دسترس است.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Locate the project directory
current_dir = Path.cwd()

if (current_dir / "data" / "processed").exists():
    PROJECT_DIR = current_dir
elif (current_dir.parent / "data" / "processed").exists():
    PROJECT_DIR = current_dir.parent
else:
    raise FileNotFoundError("Could not locate the project directory.")

# Load the final analytic dataset
data_file = PROJECT_DIR / "data" / "processed" / "nhanes_final_analytic_sample.csv"

df = pd.read_csv(data_file)

print("Dataset loaded successfully.")
print("Number of participants:", len(df))
print("Number of variables:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully.
Number of participants: 5805
Number of variables: 32

Column names:
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4', 'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU', 'INDFMPIR', 'BMXBMI', 'Mean_Systolic_BP', 'Valid_Systolic_Readings', 'Age_Group', 'Sex']


## ۲. بررسی متغیرهای موردنیاز

در این بخش، متغیرهای اصلی مورد استفاده در تحلیل احتمال را بررسی می‌کنیم. هدف، شناخت مقادیر متغیرهای دسته‌ای، بررسی مقادیر گمشده و اطمینان از آماده بودن داده‌ها برای محاسبات بعدی است.

In [2]:
# Check categorical variables
print("Sex distribution:")
print(df["Sex"].value_counts(dropna=False))

print("\nAge group distribution:")
print(df["Age_Group"].value_counts(dropna=False))

# Check missing values in key variables
key_variables = [
    "Sex",
    "Age_Group",
    "BMXBMI",
    "Mean_Systolic_BP",
    "WTMEC2YR"
]

print("\nMissing values:")
print(df[key_variables].isna().sum())

# Summary statistics for continuous variables
print("\nContinuous variables summary:")
print(df[["BMXBMI", "Mean_Systolic_BP"]].describe())

Sex distribution:
Sex
Female    3186
Male      2619
Name: count, dtype: int64

Age group distribution:
Age_Group
60+      2655
40-59    1668
20-39    1482
Name: count, dtype: int64

Missing values:
Sex                 0
Age_Group           0
BMXBMI              0
Mean_Systolic_BP    0
WTMEC2YR            0
dtype: int64

Continuous variables summary:
            BMXBMI  Mean_Systolic_BP
count  5805.000000       5805.000000
mean     29.730164        122.869308
std       7.149446         17.989238
min      11.100000         70.000000
25%      24.700000        110.666667
50%      28.400000        120.333333
75%      33.500000        132.666667
max      68.900000        215.000000


## ۳. تعریف رویدادها و متغیرهای تصادفی

### ۳.۱. منطق انتخاب رویدادها و متغیرها

با توجه به سؤال اصلی پژوهش، یعنی بررسی رابطه بین شاخص توده بدنی (BMI) و فشار خون سیستولیک در بزرگسالان، لازم است برخی ویژگی‌های افراد را به زبان احتمال و متغیرهای تصادفی تعریف کنیم.

در این تحلیل، فشار خون سیستولیک به‌عنوان پیامد اصلی مورد توجه قرار گرفته است. همچنین جنسیت و گروه سنی به‌عنوان ویژگی‌های جمعیت‌شناختی انتخاب شده‌اند تا بتوان احتمال وقوع پیامد موردنظر را در زیرگروه‌های مختلف بررسی کرد.

برای تبدیل فشار خون سیستولیک به یک پیامد دودویی، آستانه ۱۳۰ میلی‌متر جیوه در نظر گرفته شده است. بنابراین، افرادی که میانگین فشار خون سیستولیک آن‌ها حداقل ۱۳۰ باشد، در گروه دارای فشار خون سیستولیک بالا قرار می‌گیرند.

این آستانه یک تعریف عملیاتی برای تحلیل آماری حاضر است و به‌تنهایی به معنای تشخیص بالینی پرفشاری خون نیست؛ زیرا تشخیص بالینی می‌تواند به اندازه‌گیری‌های متعدد، فشار خون دیاستولیک و شرایط بالینی فرد نیز وابسته باشد.

انتخاب جنسیت و گروه سنی نیز به ما امکان می‌دهد احتمال وقوع این پیامد را در زیرگروه‌های مختلف مقایسه کنیم. در ادامه، از این رویدادها برای محاسبه احتمال‌های حاشیه‌ای، مشترک و شرطی و بررسی مفهوم استقلال استفاده خواهد شد.

### ۳.۲. تعریف رویدادها

با توجه به متغیرهای موجود در مجموعه داده، رویدادهای زیر تعریف می‌شوند:

- رویداد A: میانگین فشار خون سیستولیک فرد حداقل ۱۳۰ میلی‌متر جیوه باشد.
- رویداد B: شرکت‌کننده زن باشد.
- رویداد C: شرکت‌کننده در گروه سنی ۶۰ سال به بالا قرار داشته باشد.

این رویدادها بر اساس اطلاعات هر شرکت‌کننده در نمونه تحلیلی نهایی تعریف شده‌اند. بنابراین هر مشاهده، متناظر با یک شرکت‌کننده است.

### ۳.۳. تعریف متغیرهای تصادفی

متغیرهای تصادفی مورد استفاده در این فاز عبارت‌اند از:

- متغیر تصادفی X: میانگین فشار خون سیستولیک فرد بر حسب میلی‌متر جیوه؛ یک متغیر کمی پیوسته.
- متغیر تصادفی Y: شاخص توده بدنی فرد بر حسب کیلوگرم بر متر مربع؛ یک متغیر کمی پیوسته.
- متغیر تصادفی Z: وقوع یا عدم وقوع رویداد A؛ یک متغیر تصادفی برنولی که در صورت داشتن فشار خون سیستولیک حداقل ۱۳۰ مقدار ۱ و در غیر این صورت مقدار ۰ می‌گیرد.

تعریف متغیر Z به ما اجازه می‌دهد پیامد اصلی پژوهش را به‌صورت یک متغیر دودویی بررسی کنیم و احتمال وقوع آن را در گروه‌های مختلف محاسبه کنیم.

در بخش‌های بعدی، این متغیرها مبنای محاسبه احتمال‌های تجربی، بررسی توزیع‌های احتمالی، محاسبه امید ریاضی و واریانس و مطالعه کوواریانس خواهند بود.

In [3]:
# Define categorical events and random variables

# Event A: Systolic blood pressure is at least 130 mmHg
df["High_Systolic_BP"] = (
    df["Mean_Systolic_BP"] >= 130
).astype(int)

# Event B: Participant is female
df["Female"] = (
    df["Sex"] == "Female"
).astype(int)

# Event C: Participant is aged 60 or older
df["Age_60_Plus"] = (
    df["Age_Group"] == "60+"
).astype(int)

# Continuous random variables
X = df["Mean_Systolic_BP"]
Y = df["BMXBMI"]

# Binary random variable
Z = df["High_Systolic_BP"]

print("Event definitions:")
print("A: Systolic blood pressure >= 130 mmHg")
print("B: Female participant")
print("C: Age 60 or older")

print("\nEvent frequencies:")
print(df[["High_Systolic_BP", "Female", "Age_60_Plus"]].sum())

print("\nBinary random variable Z:")
print(Z.value_counts().sort_index())

Event definitions:
A: Systolic blood pressure >= 130 mmHg
B: Female participant
C: Age 60 or older

Event frequencies:
High_Systolic_BP    1720
Female              3186
Age_60_Plus         2655
dtype: int64

Binary random variable Z:
High_Systolic_BP
0    4085
1    1720
Name: count, dtype: int64


## ۴. محاسبه احتمال‌های حاشیه‌ای و مشترک

در این بخش، احتمال تجربی چند رویداد و احتمال وقوع هم‌زمان دو رویداد را محاسبه می‌کنیم. از آنجا که داده‌های ما مربوط به یک نمونه واقعی از شرکت‌کنندگان NHANES هستند، احتمال‌ها را با استفاده از فراوانی نسبی رویدادها در نمونه تحلیلی به دست می‌آوریم.

برای این منظور، دو رویداد زیر را در نظر می‌گیریم:

- رویداد A: میانگین فشار خون سیستولیک فرد حداقل ۱۳۰ میلی‌متر جیوه باشد.
- رویداد B: شرکت‌کننده زن باشد.

احتمال حاشیه‌ای هر رویداد، از تقسیم تعداد افراد دارای آن ویژگی بر کل تعداد افراد نمونه به دست می‌آید. احتمال مشترک دو رویداد نیز نشان می‌دهد چه نسبتی از کل شرکت‌کنندگان، هم‌زمان هر دو ویژگی را دارند.

محاسبه این احتمال‌ها به ما کمک می‌کند توزیع اولیه این ویژگی‌ها را در نمونه بشناسیم و در بخش‌های بعدی، احتمال‌های شرطی و مفهوم استقلال را بررسی کنیم.

In [4]:
# Calculate marginal and joint probabilities

n = len(df)

# Marginal probabilities
p_A = df["High_Systolic_BP"].mean()
p_B = df["Female"].mean()

# Joint probability
p_A_and_B = (
    (df["High_Systolic_BP"] == 1) &
    (df["Female"] == 1)
).mean()

print("Marginal and joint probabilities:")
print(f"P(A): {p_A:.4f}")
print(f"P(B): {p_B:.4f}")
print(f"P(A and B): {p_A_and_B:.4f}")

# Contingency table
contingency_table = pd.crosstab(
    df["High_Systolic_BP"],
    df["Female"],
    margins=True
)

print("\nContingency table:")
print(contingency_table)

Marginal and joint probabilities:
P(A): 0.2963
P(B): 0.5488
P(A and B): 0.1494

Contingency table:
Female               0     1   All
High_Systolic_BP                  
0                 1766  2319  4085
1                  853   867  1720
All               2619  3186  5805


### ۴.۲. تفسیر احتمال‌های حاشیه‌ای و مشترک

بر اساس نتایج به‌دست‌آمده، احتمال تجربی رویداد A برابر با ۰٫۲۹۶۳ است. به بیان دیگر، حدود ۲۹٫۶۳ درصد از افراد نمونه دارای میانگین فشار خون سیستولیک حداقل ۱۳۰ میلی‌متر جیوه هستند.

احتمال رویداد B برابر با ۰٫۵۴۸۸ محاسبه شده است؛ بنابراین حدود ۵۴٫۸۸ درصد از شرکت‌کنندگان نمونه زن هستند.

همچنین احتمال مشترک دو رویداد A و B برابر با ۰٫۱۴۹۴ است. این مقدار نشان می‌دهد که حدود ۱۴٫۹۴ درصد از کل افراد نمونه، هم زن هستند و هم میانگین فشار خون سیستولیک آن‌ها حداقل ۱۳۰ میلی‌متر جیوه است.

جدول فراوانی نیز نشان می‌دهد که از میان ۳۱۸۶ زن حاضر در نمونه، ۸۶۷ نفر در گروه دارای فشار خون سیستولیک حداقل ۱۳۰ قرار دارند.

این نتایج توصیفی هستند و به‌تنهایی نشان نمی‌دهند که جنسیت و فشار خون سیستولیک مستقل یا وابسته‌اند. برای بررسی دقیق‌تر این موضوع، در بخش‌های بعدی احتمال‌های شرطی و مفهوم استقلال را مطالعه خواهیم کرد.

همچنین باید توجه داشت که این احتمال‌ها بر اساس فراوانی‌های نمونه محاسبه شده‌اند و به دلیل طرح نمونه‌گیری پیچیده NHANES، لزوماً با احتمال‌های متناظر در کل جمعیت آمریکا برابر نیستند.